# Joint disease-conditioned GCN prioritization

This notebook trains one shared GCN encoder across all diseases. Each training sample supplies the same PPI graph, a disease-specific seed indicator, and a disease ID. A learned disease embedding is combined with every gene representation before producing one score per gene.

Known genes are split into outer training and held-out test sets. Only outer-training genes can become visible seeds or positive labels; held-out genes are also excluded from sampled negatives.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if not (project_root / 'bioGraph').is_dir():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from bioGraph.data.loading import load_disease_genes, load_ppi_graph
from bioGraph.gcn_prioritization import evaluate_all_diseases, predict_from_seed_genes, train_all_diseases

## Load the graph and disease associations

The processed subgraph keeps this example practical on a laptop. Replace `ppi_path` with `data/raw/PPI202207.txt` to train on the complete PPI network.

In [ ]:
ppi_path = project_root / 'data' / 'processed' / 'subgraph_5377.txt'
disease_path = project_root / 'data' / 'raw' / 'pcbi.1004120.s004.txt'

graph = load_ppi_graph(ppi_path)
diseases = load_disease_genes(disease_path)

print(f'Graph: {graph.number_of_nodes():,} genes, {graph.number_of_edges():,} interactions')
print(f'Diseases: {len(diseases)}')

## Train one model jointly across all diseases

Training returns one shared encoder and one disease-embedding table. There is no encoder pretraining and no per-disease fine-tuning. Evaluation is run separately below to produce rankings for every disease.

In [ ]:
trained = train_all_diseases(
    graph,
    diseases,
    hidden_dim=32*4,
    disease_embedding_dim=16*2,
    epochs=50,
    learning_rate=0.01,
    weight_decay=1e-4,
    negative_ratio=5,
    train_fraction=0.75,
    inner_seed_fraction=2/3,
    seed=0,
    task_batch_size=16,
)

result = evaluate_all_diseases(trained)
model = result['model']
disease_results = result['disease_results']
print(f"Finished {len(result['losses'])} epochs on {result['device']}")
print(f"Final pairwise loss: {result['losses'][-1]:.4f}")

## Inspect held-out performance

Each row is computed against that disease's held-out test genes. Training genes are excluded from its final candidate ranking.

In [ ]:
metric_rows = [
    {'disease': name, **details['metrics']}
    for name, details in disease_results.items()
]
metrics_by_disease = pd.DataFrame(metric_rows).set_index('disease')
display(metrics_by_disease)
display(metrics_by_disease.mean().rename('mean across diseases'))

## Inspect one disease and run a conditioned query

Inference must use the disease ID that selects the learned embedding. The query ranking excludes the supplied seed genes.

In [ ]:
disease_name = 'breast neoplasms'
details = disease_results[disease_name]
display(pd.Series(details['metrics'], name=disease_name))
display(pd.DataFrame(details['ranking'][:10]))

query_seed_genes = details['train_genes'][:10]
query_ranking = predict_from_seed_genes(
    model,
    result['graph_data'],
    query_seed_genes,
    disease_id=result['disease_to_id'][disease_name],
)
pd.DataFrame(query_ranking[:20])